# OMERO Data and Metadata Functions

This notebook provides functions to:
- Connect to OMERO server
- Upload images and datasets to OMERO
- Download images and datasets from OMERO
- Get and set metadata (key-value pairs, annotations, tags)

In [11]:
import omero
from omero.gateway import BlitzGateway
from omero.model import DatasetI, ProjectI, ImageI
from omero.rtypes import rstring, rlong, rint
import numpy as np
from PIL import Image
import os
from typing import Optional, Dict, List, Tuple
from pathlib import Path
from dotenv import load_dotenv

## Connection Functions

In [ ]:
def connect_to_omero(host: str, username: str, password: str, port: int = 4064) -> BlitzGateway:
    conn = BlitzGateway(username, password, host=host, port=port, secure=True)
    if not conn.connect():
        raise ConnectionError("Failed to connect to OMERO server")
    return conn

def disconnect_from_omero(conn: BlitzGateway):
    if conn:
        conn.close()

## Upload Functions (Put Data INTO OMERO)

In [ ]:
def upload_image(conn: BlitzGateway, image_path: str, dataset_id: Optional[int] = None) -> int:
    from omero.cli import CLI
    cli = CLI()
    cli.loadplugins()
    
    import_args = ["import", "-s", conn.host, "-k", conn.getSession().getUuid().val]
    if dataset_id:
        import_args.extend(["-d", str(dataset_id)])
    import_args.append(image_path)
    
    cli.invoke(import_args, strict=True)
    
    if dataset_id:
        dataset = conn.getObject("Dataset", dataset_id)
        images = list(dataset.listChildren())
        if images:
            return images[-1].getId()
    return -1

In [ ]:
def create_dataset(conn: BlitzGateway, name: str, description: Optional[str] = None, project_id: Optional[int] = None) -> int:
    dataset = omero.model.DatasetI()
    dataset.setName(rstring(name))
    if description:
        dataset.setDescription(rstring(description))
    dataset = conn.getUpdateService().saveAndReturnObject(dataset)
    dataset_id = dataset.getId().getValue()
    
    if project_id:
        link = omero.model.ProjectDatasetLinkI()
        link.setParent(omero.model.ProjectI(project_id, False))
        link.setChild(omero.model.DatasetI(dataset_id, False))
        conn.getUpdateService().saveObject(link)
    
    return dataset_id

def create_project(conn: BlitzGateway, name: str, description: Optional[str] = None) -> int:
    project = omero.model.ProjectI()
    project.setName(rstring(name))
    if description:
        project.setDescription(rstring(description))
    project = conn.getUpdateService().saveAndReturnObject(project)
    return project.getId().getValue()

## Download Functions (Get Data OUT OF OMERO)

In [ ]:
def download_image(conn: BlitzGateway, image_id: int, output_path: str) -> str:
    image = conn.getObject("Image", image_id)
    if not image:
        raise ValueError(f"Image {image_id} not found")
    
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    fileset = image.getFileset()
    if fileset:
        for orig_file in fileset.listFiles():
            raw_file_store = conn.createRawFileStore()
            try:
                raw_file_store.setFileId(orig_file.getId().getValue())
                with open(output_path, 'wb') as f:
                    offset = 0
                    size = orig_file.getSize().getValue()
                    chunk_size = 1024 * 1024
                    while offset < size:
                        chunk = raw_file_store.read(offset, min(chunk_size, size - offset))
                        f.write(chunk)
                        offset += len(chunk)
                return str(output_path)
            finally:
                raw_file_store.close()
    
    pixels = image.getPrimaryPixels()
    planes = [pixels.getPlane(z, c, t) 
              for z in range(image.getSizeZ()) 
              for c in range(image.getSizeC()) 
              for t in range(image.getSizeT())]
    
    img = Image.fromarray(planes[0])
    if len(planes) > 1:
        img.save(output_path, save_all=True, append_images=[Image.fromarray(p) for p in planes[1:]])
    else:
        img.save(output_path)
    return str(output_path)

def download_dataset(conn: BlitzGateway, dataset_id: int, output_dir: str) -> List[str]:
    dataset = conn.getObject("Dataset", dataset_id)
    if not dataset:
        raise ValueError(f"Dataset {dataset_id} not found")
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    downloaded_files = []
    for image in dataset.listChildren():
        safe_name = "".join(c for c in image.getName() if c.isalnum() or c in (' ', '.', '_', '-'))
        output_path = output_dir / f"{safe_name}_{image.getId()}.tiff"
        try:
            downloaded_files.append(download_image(conn, image.getId(), str(output_path)))
        except Exception as e:
            print(f"Error downloading image {image.getId()}: {e}")
    
    return downloaded_files

## Metadata Functions (Get/Set Key-Value Pairs, Annotations, Tags)

In [16]:
def add_key_value_pairs(conn: BlitzGateway, object_type: str, object_id: int, 
                        key_value_data: Dict[str, str]) -> int:
    """
    Add key-value pair metadata to an OMERO object.
    
    Args:
        conn: BlitzGateway connection object
        object_type: Type of object ('Image', 'Dataset', 'Project', etc.)
        object_id: ID of the object
        key_value_data: Dictionary of key-value pairs to add
    
    Returns:
        Annotation ID
    """
    map_ann = omero.gateway.MapAnnotationWrapper(conn)
    namespace = omero.constants.metadata.NSCLIENTMAPANNOTATION
    map_ann.setNs(namespace)
    
    # Convert dict to list of [key, value] pairs
    key_value_list = [[str(k), str(v)] for k, v in key_value_data.items()]
    map_ann.setValue(key_value_list)
    map_ann.save()
    
    # Link annotation to object
    obj = conn.getObject(object_type, object_id)
    if obj:
        obj.linkAnnotation(map_ann)
        print(f"Added {len(key_value_data)} key-value pairs to {object_type} {object_id}")
        return map_ann.getId()
    else:
        raise ValueError(f"{object_type} {object_id} not found")


def get_key_value_pairs(conn: BlitzGateway, object_type: str, object_id: int) -> Dict[str, str]:
    """
    Get all key-value pair metadata from an OMERO object.
    
    Args:
        conn: BlitzGateway connection object
        object_type: Type of object ('Image', 'Dataset', 'Project', etc.)
        object_id: ID of the object
    
    Returns:
        Dictionary of all key-value pairs
    """
    obj = conn.getObject(object_type, object_id)
    if not obj:
        raise ValueError(f"{object_type} {object_id} not found")
    
    all_key_values = {}
    for ann in obj.listAnnotations():
        if isinstance(ann, omero.gateway.MapAnnotationWrapper):
            for key, value in ann.getValue():
                all_key_values[key] = value
    
    print(f"Retrieved {len(all_key_values)} key-value pairs from {object_type} {object_id}")
    return all_key_values


def add_tag(conn: BlitzGateway, object_type: str, object_id: int, tag_text: str, 
            tag_description: Optional[str] = None) -> int:
    """
    Add a tag to an OMERO object.
    
    Args:
        conn: BlitzGateway connection object
        object_type: Type of object ('Image', 'Dataset', 'Project', etc.)
        object_id: ID of the object
        tag_text: Text of the tag
        tag_description: Optional description for the tag
    
    Returns:
        Tag annotation ID
    """
    tag_ann = omero.gateway.TagAnnotationWrapper(conn)
    tag_ann.setValue(tag_text)
    if tag_description:
        tag_ann.setDescription(tag_description)
    tag_ann.save()
    
    # Link tag to object
    obj = conn.getObject(object_type, object_id)
    if obj:
        obj.linkAnnotation(tag_ann)
        print(f"Added tag '{tag_text}' to {object_type} {object_id}")
        return tag_ann.getId()
    else:
        raise ValueError(f"{object_type} {object_id} not found")


def get_tags(conn: BlitzGateway, object_type: str, object_id: int) -> List[str]:
    """
    Get all tags from an OMERO object.
    
    Args:
        conn: BlitzGateway connection object
        object_type: Type of object ('Image', 'Dataset', 'Project', etc.)
        object_id: ID of the object
    
    Returns:
        List of tag texts
    """
    obj = conn.getObject(object_type, object_id)
    if not obj:
        raise ValueError(f"{object_type} {object_id} not found")
    
    tags = []
    for ann in obj.listAnnotations():
        if isinstance(ann, omero.gateway.TagAnnotationWrapper):
            tags.append(ann.getValue())
    
    print(f"Retrieved {len(tags)} tags from {object_type} {object_id}")
    return tags


def add_comment(conn: BlitzGateway, object_type: str, object_id: int, comment_text: str) -> int:
    """
    Add a comment to an OMERO object.
    
    Args:
        conn: BlitzGateway connection object
        object_type: Type of object ('Image', 'Dataset', 'Project', etc.)
        object_id: ID of the object
        comment_text: Text of the comment
    
    Returns:
        Comment annotation ID
    """
    comment_ann = omero.gateway.CommentAnnotationWrapper(conn)
    comment_ann.setValue(comment_text)
    comment_ann.save()
    
    # Link comment to object
    obj = conn.getObject(object_type, object_id)
    if obj:
        obj.linkAnnotation(comment_ann)
        print(f"Added comment to {object_type} {object_id}")
        return comment_ann.getId()
    else:
        raise ValueError(f"{object_type} {object_id} not found")


def get_comments(conn: BlitzGateway, object_type: str, object_id: int) -> List[str]:
    """
    Get all comments from an OMERO object.
    
    Args:
        conn: BlitzGateway connection object
        object_type: Type of object ('Image', 'Dataset', 'Project', etc.)
        object_id: ID of the object
    
    Returns:
        List of comment texts
    """
    obj = conn.getObject(object_type, object_id)
    if not obj:
        raise ValueError(f"{object_type} {object_id} not found")
    
    comments = []
    for ann in obj.listAnnotations():
        if isinstance(ann, omero.gateway.CommentAnnotationWrapper):
            comments.append(ann.getValue())
    
    print(f"Retrieved {len(comments)} comments from {object_type} {object_id}")
    return comments

In [17]:
def get_image_metadata(conn: BlitzGateway, image_id: int) -> Dict:
    """
    Get comprehensive metadata for an image.
    
    Args:
        conn: BlitzGateway connection object
        image_id: ID of the image
    
    Returns:
        Dictionary containing all metadata
    """
    image = conn.getObject("Image", image_id)
    if not image:
        raise ValueError(f"Image {image_id} not found")
    
    metadata = {
        'id': image.getId(),
        'name': image.getName(),
        'description': image.getDescription(),
        'acquisition_date': str(image.getAcquisitionDate()) if image.getAcquisitionDate() else None,
        'owner': image.getOwnerFullName(),
        'dimensions': {
            'size_x': image.getSizeX(),
            'size_y': image.getSizeY(),
            'size_z': image.getSizeZ(),
            'size_c': image.getSizeC(),
            'size_t': image.getSizeT(),
        },
        'pixel_size': {
            'x': image.getPixelSizeX(),
            'y': image.getPixelSizeY(),
            'z': image.getPixelSizeZ(),
        },
        'channels': [],
        'key_value_pairs': {},
        'tags': [],
        'comments': [],
    }
    
    # Get channel information
    for channel in image.getChannels():
        channel_info = {
            'label': channel.getLabel(),
            'color': channel.getColor().getHtml() if channel.getColor() else None,
            'wavelength': channel.getEmissionWave(),
        }
        metadata['channels'].append(channel_info)
    
    # Get annotations
    for ann in image.listAnnotations():
        if isinstance(ann, omero.gateway.MapAnnotationWrapper):
            for key, value in ann.getValue():
                metadata['key_value_pairs'][key] = value
        elif isinstance(ann, omero.gateway.TagAnnotationWrapper):
            metadata['tags'].append(ann.getValue())
        elif isinstance(ann, omero.gateway.CommentAnnotationWrapper):
            metadata['comments'].append(ann.getValue())
    
    return metadata


def get_dataset_info(conn: BlitzGateway, dataset_id: int) -> Dict:
    """
    Get information about a dataset including all its images.
    
    Args:
        conn: BlitzGateway connection object
        dataset_id: ID of the dataset
    
    Returns:
        Dictionary containing dataset information
    """
    dataset = conn.getObject("Dataset", dataset_id)
    if not dataset:
        raise ValueError(f"Dataset {dataset_id} not found")
    
    info = {
        'id': dataset.getId(),
        'name': dataset.getName(),
        'description': dataset.getDescription(),
        'owner': dataset.getOwnerFullName(),
        'image_count': dataset.countChildren(),
        'images': [],
        'key_value_pairs': {},
        'tags': [],
    }
    
    # Get image IDs and names
    for image in dataset.listChildren():
        info['images'].append({
            'id': image.getId(),
            'name': image.getName(),
        })
    
    # Get annotations
    for ann in dataset.listAnnotations():
        if isinstance(ann, omero.gateway.MapAnnotationWrapper):
            for key, value in ann.getValue():
                info['key_value_pairs'][key] = value
        elif isinstance(ann, omero.gateway.TagAnnotationWrapper):
            info['tags'].append(ann.getValue())
    
    return info

## Example Usage

Below are examples of how to use these functions:

### 1. Connect to OMERO

In [18]:
# Connect to OMERO server
load_dotenv("defualt.env")
load_dotenv(".env", override=True)

HOST = os.getenv('OMERO_HOST')
USERNAME = os.getenv('OMERO_USERNAME')
PASSWORD = os.getenv('OMERO_PASSWORD')

conn = connect_to_omero(HOST, USERNAME, PASSWORD)

2025-10-20 16:52:37,039 DEBUG [                           omero.gateway] (MainThread) localhost
2025-10-20 16:52:37,041 DEBUG [                           omero.gateway] (MainThread) 4064
2025-10-20 16:52:37,041 DEBUG [                           omero.gateway] (MainThread) []
2025-10-20 16:52:37,068 DEBUG [                           omero.gateway] (MainThread) Connect attempt, sUuid=None, group=None, self.sUuid=None
2025-10-20 16:52:37,068 DEBUG [                           omero.gateway] (MainThread) Creating Session...
2025-10-20 16:52:37,513 DEBUG [                     omero.gateway.utils] (MainThread) Setting 'omero.client.uuid' to '9c581582-e2c0-4571-a88d-d9cb98f39dff'
2025-10-20 16:52:37,513 DEBUG [                     omero.gateway.utils] (MainThread) Setting 'omero.event' to 'Internal'
2025-10-20 16:52:37,514 DEBUG [                     omero.gateway.utils] (MainThread) Setting 'omero.session.uuid' to '86b9e2b7-550b-442c-a067-57a22a58d45e'
2025-10-20 16:52:37,514 DEBUG [         

Connected to OMERO server: localhost
User: root


### 2. Upload Data to OMERO

In [19]:
# Create a project and dataset
project_id = create_project(conn, "My Research Project", "Description of the project")
dataset_id = create_dataset(conn, "Experiment 1", "First experiment data", project_id=project_id)

# Upload an image to the dataset
image_id = upload_image(conn, "/path/to/image.tif", dataset_id=dataset_id, 
                        image_name="Sample Image", description="Control sample")

print(f"Project ID: {project_id}")
print(f"Dataset ID: {dataset_id}")
print(f"Image ID: {image_id}")

Created project 'My Research Project' (ID: 1)
Created dataset 'Experiment 1' (ID: 1) in project 1
Deprecated warning: use 'omero --debug=x [args]' to debug
Running omero with debugging == 1


ModuleNotFoundError: No module named 'omero.plugins.import_'

### 3. Add Metadata to OMERO Objects

In [ ]:
# Add key-value pairs to an image
metadata = {
    "experiment_type": "fluorescence microscopy",
    "magnification": "60x",
    "exposure_time": "100ms",
    "temperature": "37C",
    "researcher": "John Doe"
}
add_key_value_pairs(conn, "Image", image_id, metadata)

# Add tags
add_tag(conn, "Image", image_id, "control")
add_tag(conn, "Image", image_id, "high-quality")

# Add a comment
add_comment(conn, "Image", image_id, "This image shows excellent cell morphology")

### 4. Retrieve Metadata from OMERO

In [ ]:
# Get all key-value pairs
kv_pairs = get_key_value_pairs(conn, "Image", image_id)
print("Key-Value Pairs:", kv_pairs)

# Get all tags
tags = get_tags(conn, "Image", image_id)
print("Tags:", tags)

# Get all comments
comments = get_comments(conn, "Image", image_id)
print("Comments:", comments)

# Get comprehensive image metadata
full_metadata = get_image_metadata(conn, image_id)
print("\nFull Image Metadata:")
import json
print(json.dumps(full_metadata, indent=2))

### 5. Download Data from OMERO

In [ ]:
# Download a single image
download_image(conn, image_id, "/path/to/save/downloaded_image.tif")

# Download all images in a dataset
downloaded_files = download_dataset(conn, dataset_id, "/path/to/save/directory/")
print(f"Downloaded {len(downloaded_files)} files")

### 6. Get Dataset Information

In [ ]:
# Get dataset information
dataset_info = get_dataset_info(conn, dataset_id)
print("Dataset Information:")
print(json.dumps(dataset_info, indent=2))

### 7. Disconnect from OMERO

In [ ]:
# Always disconnect when done
disconnect_from_omero(conn)